# Streaming QA Generation

Generate real-time QA at different time points within a clip.
Only frames/ASR up to the current timestamp are given to GPT-4o.
Includes post-hoc verification for future information leakage.

In [1]:
import sys, os
os.chdir("..")
sys.path.append("scripts")
os.environ["OPENAI_API_KEY"] = "REPLACED_BY_ENV_LOADER_SEE_README"  # <-- PUT YOUR KEY HERE

from qa_generation import extract_clip_frames, frame_to_base64
from pathlib import Path
import json, time, re
from openai import OpenAI

In [2]:
VIDEO = "UltrasoundCrawler_KeyCode_20260323_v2/output/20260520_162816_youtube/media/case_reasoning/8V649L5Q368.mp4"
CLIPS_FILE = "results/clips/8V649L5Q368_clips.json"

with open(CLIPS_FILE) as f:
    clips_data = json.load(f)
clips = clips_data["clips"]

test_clip = clips[1]
print(f"Clip {test_clip['clip_idx']}: {test_clip['start']:.0f}-{test_clip['end']:.0f}s ({test_clip['duration']:.0f}s)")
print(f"Topic: {test_clip.get('topic', 'N/A')}")


Clip 1: 87-250s (163s)
Topic: Diagnosis of pneumothorax using ultrasound


In [6]:
STREAMING_QA_PROMPT = """You are a senior ultrasound instructor and medical education expert.

Given video frames seen SO FAR (up to {current_time:.0f}s) and the speech transcription heard so far, generate exactly 5 question-answer pairs from different perspectives.

CRITICAL CONSTRAINT: The viewer has ONLY seen the video up to {current_time:.0f}s. They CANNOT see anything after this point. All answers must be based ONLY on what has been shown/heard so far.

IMPORTANT: These frames are from a VIDEO (shown in temporal order). Describe actions, movements, and temporal changes observed so far.

Frames seen: {n_frames} frames from {start_time:.0f}s to {current_time:.0f}s
ASR heard so far: "{asr_so_far}"
Topic: {topic}

QA Types (generate exactly one of each):
1. "scene_description" - What is happening in the video so far? Describe the scanning process and visible structures up to this point.
2. "sonographer_intent" - What is the operator currently trying to do or find? What is their diagnostic goal at this moment?
3. "next_action_guidance" - Based on what has been seen so far, what should the sonographer do next, or what area/structure should be examined next and why, or how should the probe be adjusted?
4. "knowledge" - What medical knowledge is relevant to what's being shown/discussed so far? Include clinical significance.
5. "fine_grained" - Describe specific visual attributes currently visible: anatomical landmarks, echogenicity patterns, probe positioning technique.

Output ONLY a JSON array of 5 objects:
[
  {{"type": "scene_description", "timestamp": {current_time}, "question": "What has been demonstrated so far in this ultrasound clip?", "answer": "..."}},
  {{"type": "sonographer_intent", "timestamp": {current_time}, "question": "...", "answer": "..."}},
  {{"type": "next_action_guidance", "timestamp": {current_time}, "question": "...", "answer": "..."}},
  {{"type": "knowledge", "timestamp": {current_time}, "question": "...", "answer": "..."}},
  {{"type": "fine_grained", "timestamp": {current_time}, "question": "...", "answer": "..."}}
]

Important rules:
- All answers based ONLY on what's been seen up to {current_time:.0f}s
- Questions should be educational and clinically relevant
- Answers should be detailed but concise (2-4 sentences)
- Reference temporal aspects: "so far...", "at this point...", "the operator has been..."
- Output ONLY valid JSON array, no other text"""

def generate_streaming_qa(video_path, clip, ratio=0.5):
    client = OpenAI()
    current_time = clip['start'] + clip['duration'] * ratio
    
    # Only frames up to current_time
    frames = extract_clip_frames(str(video_path), clip['start'], current_time, num_frames=4)
    
    # Only ASR up to current_time
    full_text = clip.get('text', '')
    words = full_text.split()
    asr_so_far = ' '.join(words[:int(len(words) * ratio)])
    
    prompt = STREAMING_QA_PROMPT.format(
        current_time=current_time,
        start_time=clip['start'],
        n_frames=len(frames),
        asr_so_far=asr_so_far[:1500],
        topic=clip.get('topic', 'ultrasound')
    )
    
    content = []
    for frame in frames:
        b64 = frame_to_base64(frame)
        content.append({"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{b64}", "detail": "low"}})
    content.append({"type": "text", "text": prompt})
    
    t0 = time.time()
    response = client.chat.completions.create(
        model="gpt-4o",
        messages=[{"role": "user", "content": content}],
        temperature=0.3, max_tokens=800,
    )
    elapsed = time.time() - t0
    raw = response.choices[0].message.content
    tokens = response.usage.total_tokens if response.usage else 0
    print(f"  t={current_time:.0f}s | {elapsed:.1f}s | {tokens} tokens")
    
    try:
        qa_pairs = json.loads(raw)
    except:
        m = re.search(r'\[.*\]', raw, re.DOTALL)
        qa_pairs = json.loads(m.group(0)) if m else []
    
    for qa in qa_pairs:
        qa['clip_idx'] = clip['clip_idx']
        qa['query_time'] = round(current_time, 2)
        qa['ratio'] = ratio
    
    return qa_pairs

In [7]:
all_streaming_qa = []

for ratio in [0.25, 0.5, 0.75]:
    print(f"\n--- {int(ratio*100)}% of clip ---")
    qa_pairs = generate_streaming_qa(VIDEO, test_clip, ratio=ratio)
    all_streaming_qa.extend(qa_pairs)
    for qa in qa_pairs:
        print(f"  [{qa['type']}] Q: {qa['question']}")
        print(f"    A: {qa['answer'][:80]}...")

print(f"\nTotal: {len(all_streaming_qa)} streaming QA")


--- 25% of clip ---
  t=128s | 4.6s | 1476 tokens
  [scene_description] Q: What has been demonstrated so far in this ultrasound clip?
    A: The operator is preparing to perform an ultrasound scan to diagnose pneumothorax...
  [sonographer_intent] Q: What is the operator currently trying to do or find?
    A: The operator is attempting to position the probe correctly to visualize the pleu...
  [next_action_guidance] Q: What should the sonographer do next to improve the examination?
    A: The sonographer should continue scanning across different rib spaces to ensure c...
  [knowledge] Q: What medical knowledge is relevant to what's being shown so far?
    A: Understanding the concept of pleural sliding is essential, as its presence or ab...
  [fine_grained] Q: Describe specific visual attributes currently visible in the ultrasound.
    A: The ultrasound screen shows a linear probe image with rib shadows and soft tissu...

--- 50% of clip ---
  t=169s | 4.3s | 1662 tokens
  [scene_desc

In [9]:
def verify_question_validity_vlm(qa, clip, video_path):
    """
    VLM verification: Give GPT-4o Vision the frames BEFORE and AFTER the question timestamp.
    Check if the question could be asked based only on the "before" frames.
    """
    client = OpenAI()
    ratio = qa['ratio']
    current_time = qa['query_time']
    
    # Frames BEFORE current_time (what the viewer has seen)
    frames_before = extract_clip_frames(str(video_path), clip['start'], current_time, num_frames=3)
    
    # Frames AFTER current_time (what the viewer has NOT seen)
    frames_after = extract_clip_frames(str(video_path), current_time, clip['end'], num_frames=3)
    
    # Build message
    content = []
    
    # Add "before" frames
    content.append({"type": "text", "text": "=== FRAMES ALREADY SEEN (before the question) ==="})
    for frame in frames_before:
        b64 = frame_to_base64(frame)
        content.append({"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{b64}", "detail": "low"}})
    
    # Add "after" frames
    content.append({"type": "text", "text": "=== FRAMES NOT YET SEEN (after the question) ==="})
    for frame in frames_after:
        b64 = frame_to_base64(frame)
        content.append({"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{b64}", "detail": "low"}})
    
    # Add verification prompt
    verify_text = f"""A viewer is watching an ultrasound teaching video. 
At t={current_time:.0f}s, they ask this question:

Question: "{qa['question']}"

I showed you two sets of frames:
1. FRAMES ALREADY SEEN (before the question was asked)
2. FRAMES NOT YET SEEN (after the question was asked)

Determine: Could this question be naturally triggered by ONLY the "already seen" frames?
Or does it reference something that only appears in the "not yet seen" frames?

Output JSON: {{"question_valid": true/false, "reason": "brief explanation"}}"""
    
    content.append({"type": "text", "text": verify_text})
    
    response = client.chat.completions.create(
        model="gpt-4o",
        messages=[{"role": "user", "content": content}],
        temperature=0.1, max_tokens=200,
    )
    
    raw = response.choices[0].message.content
    try:
        return json.loads(raw)
    except:
        m = re.search(r'\{.*\}', raw, re.DOTALL)
        return json.loads(m.group(0)) if m else {"question_valid": None}


# Run VLM verification
print("VLM Verification (checking question validity)...\n")
for qa in all_streaming_qa:
    result = verify_question_validity_vlm(qa, test_clip, VIDEO)
    status = "VALID" if result.get('question_valid') else "INVALID"
    print(f"[{status}] t={qa['query_time']:.0f}s | [{qa['type']}] {qa['question'][:50]}")
    print(f"  Reason: {result.get('reason', '')}")
    print()

VLM Verification (checking question validity)...

[VALID] t=128s | [scene_description] What has been demonstrated so far in this ultrasou
  Reason: The question can be triggered by the 'already seen' frames, as they show the ultrasound procedure and the pleural interface, which are relevant to understanding what is being demonstrated.

[VALID] t=128s | [sonographer_intent] What is the operator currently trying to do or fin
  Reason: The question could be naturally triggered by the 'already seen' frames, as they show the operator using an ultrasound device on a patient, which suggests they are trying to find or examine something specific.

[VALID] t=128s | [next_action_guidance] What should the sonographer do next to improve the
  Reason: The question about improving the examination could naturally arise from the 'already seen' frames, as they show the sonographer performing an ultrasound and the resulting image. The viewer might be questioning the technique or image quality based on wh

In [11]:
output = {
    'video_id': clips_data['video_id'],
    'clip_idx': test_clip['clip_idx'],
    'num_streaming_qa': len(all_streaming_qa),
    'streaming_qa': all_streaming_qa,
}
Path('results/qa').mkdir(parents=True, exist_ok=True)
with open(f"results/qa/{clips_data['video_id']}_streaming_qa.json", 'w') as f:
    json.dump(output, f, indent=2, ensure_ascii=False)
print("Saved!")

Saved!
